In [17]:
import requests
import urllib3
import pandas as pd
import numpy as np

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://www.safetydata.go.kr/V2/api/DSSP-IF-00195"
serviceKey = "0YP09R0RZP9PV65C"

In [ ]:
# Initialize empty list to store all data
all_data = []

# Make API calls for pages 1-40
for page in range(1, 41):
    payloads = {
        "serviceKey": serviceKey,
        "returnType": "json",
        "pageNo": str(page),
        "numOfRows": "1000",
    }
    
    response = requests.get(url, params=payloads)
    
    # Check if response contains data
    if 'body' in response.json():
        all_data.extend(response.json()['body'])


TypeError: 'NoneType' object is not iterable

In [73]:
# Create DataFrame from all collected data
df = pd.DataFrame(all_data)

# Filter rows where MNG_INST_NM contains "대구광역시"
df = df[df['MNG_INST_NM'].str.contains("대구광역시", na=False)]

# Save full dataset to CSV
df.to_csv('daegu_disaster_safety_data.csv', index=False, encoding='utf-8')

In [74]:
# 대구광역시 북구만
df = df[df['MNG_INST_NM'].str.contains("대구광역시 북구", na=False)].reset_index(drop=True)
df.head()

,GRND_UDGD_SE,ORTM_UTLZ_TYPE,SGG_CD,FCLT_CD,FCLT_SE_CD,ROAD_NM_CD,LOT_MIN,SHNT_PSBLTY_NOPE,FCLT_SCL,LOT_SEC,...,LAT_PROVIN,LAT_MIN,MNG_INST_TELNO,FCLT_NM,FCLT_DSGN_DAY,MNG_INST_NM,FCLT_ADDR_LOTNO,SCL_UNIT,LOT_PROVIN,EMD_CD
0,1,지하주차장,3450000,S201100009,3,4235125,36,7696,6350,8,...,35,53,053-665-4310,"산격2차 청구맨션 전체동의 지하주차장 1,2층",20110325,대구광역시 북구청,대구광역시 북구 산격동 1404번지 19호,㎡,128,3450039
1,1,지하주차장,3450000,S201100010,3,3145017,36,2521,2080,5,...,35,53,053-665-4310,산격1차청구맨션 전체동의 지하주차장 1층,20110325,대구광역시 북구청,대구광역시 북구 산격동 1442번지 4호,㎡,128,3450039
2,0,생활용수,3450000,E201200001,4,3007005,36,0,210,35,...,35,53,053-665-3900,스카이 사우나 3층,20120307,대구광역시 북구청,대구광역시 북구 산격동 1238번지 1호,t(톤),128,3450038
3,1,교육시설,3450000,S200300061,3,3145021,36,1001,826,35,...,35,53,053-665-4310,경북대 도서관 신관 지하 1층,20030820,대구광역시 북구청,대구광역시 북구 산격동 1370번지 1호,㎡,128,3450038
4,1,교육시설,3450000,S200300062,3,3145021,36,1522,1256,35,...,35,53,053-665-4310,경북대 복현회관 지하 1층,20030820,대구광역시 북구청,대구광역시 북구 산격동 1370번지 1호,㎡,128,3450038


In [49]:
df.columns

Index(['GRND_UDGD_SE', 'ORTM_UTLZ_TYPE', 'SGG_CD', 'FCLT_CD', 'FCLT_SE_CD',
       'ROAD_NM_CD', 'LOT_MIN', 'SHNT_PSBLTY_NOPE', 'FCLT_SCL', 'LOT_SEC',
       'EMD_NM', 'FCLT_ADDR_RONA', 'SE_CD', 'OPN_YN', 'LAT_SEC', 'LAT_PROVIN',
       'LAT_MIN', 'MNG_INST_TELNO', 'FCLT_NM', 'FCLT_DSGN_DAY', 'MNG_INST_NM',
       'FCLT_ADDR_LOTNO', 'SCL_UNIT', 'LOT_PROVIN', 'EMD_CD'],
      dtype='object')

대피소구분코드(한파쉼터:1,무더위쉼터:2,지진옥외대피장소:3,지진해일긴급대피장소:4)	
시설구분코드

In [75]:
# Convert degrees, minutes, seconds to decimal degrees
df['경도'] = df['LOT_PROVIN'] + df['LOT_MIN']/60 + df['LOT_SEC']/3600
df['위도'] = df['LAT_PROVIN'] + df['LAT_MIN']/60 + df['LAT_SEC']/3600

# str 문자열 처리
df['FCLT_SE_CD'] = df['FCLT_SE_CD'].str.strip()

# Create mapping dictionary for facility codes
facility_type_map = {
    '1': '한파',
    '2': '무더위', 
    '3': '지진',
    '4': '지진해일'
}

# Convert facility codes before renaming
df['FCLT_SE_CD'] = df['FCLT_SE_CD'].map(facility_type_map)

# Fill zero values in SHNT_PSBLTY_NOPE with FCLT_SCL values
df.loc[df['SHNT_PSBLTY_NOPE'] == '0', 'SHNT_PSBLTY_NOPE'] = df.loc[df['SHNT_PSBLTY_NOPE'] == '0', 'FCLT_SCL']

t = df.rename(columns={
    'FCLT_SE_CD': '시설구분코드',
    'FCLT_CD': '시설코드', 
    'FCLT_NM': '시설명',
    'FCLT_SCL': '시설규모',
    'SHNT_PSBLTY_NOPE': '대피가능인원수',
    'MNG_INST_NM': '관리기관명',
    'MNG_INST_TELNO': '관리기관전화번호',
    'FCLT_ADDR_RONA': '시설주소도로명',
    'OPN_YN': '개방여부',
    'GRND_UDGD_SE': '지상지하구분'
})[['시설구분코드', '시설코드', '시설명', '시설규모', '대피가능인원수', '관리기관명', '관리기관전화번호',
     '시설주소도로명', '경도', '위도', '개방여부', '지상지하구분']]
t


,시설구분코드,시설코드,시설명,시설규모,대피가능인원수,관리기관명,관리기관전화번호,시설주소도로명,경도,위도,개방여부,지상지하구분
0,지진,S201100009,"산격2차 청구맨션 전체동의 지하주차장 1,2층",6350,7696,대구광역시 북구청,053-665-4310,"대구광역시 북구 대구체육관로4길 2 (산격동, 산격2차청구맨션)",128.602222,35.890278,Y,1
1,지진,S201100010,산격1차청구맨션 전체동의 지하주차장 1층,2080,2521,대구광역시 북구청,053-665-4310,"대구광역시 북구 대구체육관로 12 (산격동, 산격청구맨션)",128.601389,35.890556,Y,1
2,지진해일,E201200001,스카이 사우나 3층,210,210,대구광역시 북구청,053-665-3900,대구광역시 북구 동북로 156 (산격동),128.609722,35.899444,Y,0
3,지진,S200300061,경북대 도서관 신관 지하 1층,826,1001,대구광역시 북구청,053-665-4310,"대구광역시 북구 대학로 80 (산격동, 경북대학교)",128.609722,35.887500,Y,1
4,지진,S200300062,경북대 복현회관 지하 1층,1256,1522,대구광역시 북구청,053-665-4310,"대구광역시 북구 대학로 80 (산격동, 경북대학교)",128.609722,35.887500,Y,1
...,...,...,...,...,...,...,...,...,...,...,...,...
186,지진,S201600005,금호천년나무1단지 지하주차장 1층,17427,21123,대구광역시 북구청,053-665-3930,"대구광역시 북구 한강로 55 (사수동, 대구금호엘에이치천년나무1단지)",128.515556,35.897500,Y,1
187,지진,S200000008,두산위브2001 지하주차장 1층,21583,26161,대구광역시 북구청,053-665-3930,"대구광역시 북구 매천로2길 19 (팔달동, 두산위브2001)",128.544167,35.898333,Y,1
188,지진해일,E200000014,복현서한타운,150,150,대구광역시 북구청,053-665-3640,"대구광역시 북구 검단로 50 (복현동, 복현서한타운)",128.616944,35.902500,Y,0
189,지진,S202100005,복현건영아파트 전체동 지하주차장 1층,4795,5812,대구광역시 북구청,053-665-3640,"대구광역시 북구 검단로 28 (복현동, 복현건영아파트)",128.615556,35.900278,N,1


In [76]:
t.to_csv('data.csv', index=False, sep='|')